In [ ]:
import os
import torch
from PIL import Image
from torchvision import transforms
import lpips
from pytorch_msssim import ms_ssim

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Preprocessing (resize all to same size, normalize)
transform = transforms.Compose([
    transforms.Resize((256, 256)),   # match size to avoid errors
    transforms.ToTensor()
])


In [4]:
def compute_ms_ssim_lpips(real_dir, fake_dir, max_images=100):
    """Compute MS-SSIM and LPIPS between real and fake images in given dirs"""
    real_files = [os.path.join(real_dir, f) for f in os.listdir(real_dir) if f.startswith("ISIC")]
    fake_files = [os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if not f.startswith("ISIC")]

    # Balance number of images
    num_images = min(len(real_files), len(fake_files), max_images)
    real_files, fake_files = real_files[:num_images], fake_files[:num_images]

    if num_images == 0:
        return None, None

    # Load LPIPS model
    lpips_model = lpips.LPIPS(net='alex').to(device)

    ms_ssim_scores, lpips_scores = [], []
    for r, f in zip(real_files, fake_files):
        real_img = transform(Image.open(r).convert("RGB")).unsqueeze(0).to(device)
        fake_img = transform(Image.open(f).convert("RGB")).unsqueeze(0).to(device)

        # MS-SSIM
        ms_val = ms_ssim(real_img, fake_img, data_range=1.0, size_average=True).item()
        ms_ssim_scores.append(ms_val)

        # LPIPS
        lp_val = lpips_model(real_img, fake_img).item()
        lpips_scores.append(lp_val)

    return sum(ms_ssim_scores)/len(ms_ssim_scores), sum(lpips_scores)/len(lpips_scores)


In [5]:
root_dir = r"C:\\Users\\devgo\\OneDrive\\Desktop\\fid\\data\\processed_images"  # adjust path if needed

all_ms, all_lp = [], []

for cls in os.listdir(root_dir):
    class_path = os.path.join(root_dir, cls)
    if os.path.isdir(class_path):
        ms, lp = compute_ms_ssim_lpips(class_path, class_path, max_images=100)
        if ms is not None:
            print(f"Class: {cls} | MS-SSIM: {ms:.4f} | LPIPS: {lp:.4f}")
            all_ms.append(ms)
            all_lp.append(lp)

if all_ms and all_lp:
    print("\n=== Overall Results ===")
    print(f"Average MS-SSIM: {sum(all_ms)/len(all_ms):.4f}")
    print(f"Average LPIPS: {sum(all_lp)/len(all_lp):.4f}")


Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


c:\Users\devgo\OneDrive\Desktop\fid\.venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\devgo\OneDrive\Desktop\fid\.venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: c:\Users\devgo\OneDrive\Desktop\fid\.venv\Lib\site-packages\lpips\weights\v0.1\alex.pth
Class: AKIEC | MS-SSIM: 0.3307 | LPIPS: 0.2335
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: c:\Users\devgo\OneDrive\Desktop\fid\.venv\Lib\site-packages\lpips\weights\v0.1\alex.pth
Class: BCC | MS-SSIM: 0.3695 | LPIPS: 0.2553
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: c:\Users\devgo\OneDrive\Desktop\fid\.venv\Lib\site-packages\lpips\weights\v0.1\alex.pth
Class: BKL | MS-SSIM: 0.3698 | LPIPS: 0.2786
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: c:\Users\devgo\OneDrive\Desktop\fid\.venv\Lib\site-packages\lpips\weights\v0.1\alex.pth
Class: DF | MS-SSIM: 0.4124 | LPIPS: 0.2214
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]
Loading model from: c:\Users\devgo\OneDrive\Desktop\fid\.venv\Lib\site-packages\lpips\weights\v0.1\